#### Import the relivant libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

#### Loading the 8 datasets

In [3]:
import pandas as pd

PATH = "datasets/raw/"

df_adm = pd.read_csv(PATH + "admissions.csv")
df_diag = pd.read_csv(PATH + "diagnoses.csv")
df_lab = pd.read_csv(PATH + "lab_results.csv")
df_ed = pd.read_csv(PATH + "ed_visits.csv")
df_meds = pd.read_csv(PATH + "medications.csv")
df_vitals = pd.read_csv(PATH + "vitals.csv")
df_readmit = pd.read_csv(PATH + "readmissions.csv")
df_pat = pd.read_csv(PATH + "patients.csv")

### DATASET STANDARDISATION

#### Standardizing the admission data types

In [4]:
df_adm["admission_date"] = pd.to_datetime(df_adm["admission_date"])
df_adm["discharge_date"] = pd.to_datetime(df_adm["discharge_date"])

df_adm["length_of_stay_days"] = df_adm["length_of_stay_days"].astype(float)

df_adm["icu_admitted"] = df_adm["icu_admitted"].astype(int)
df_adm["icu_days"] = df_adm["icu_days"].astype(float)

df_adm["surgery_performed"] = df_adm["surgery_performed"].astype(int)
df_adm["num_procedures"] = df_adm["num_procedures"].astype(int)

df_adm["readmitted_within_30d"] = df_adm["readmitted_within_30d"].astype(int)
cat_cols = [
    "admission_type",
    "admission_source",
    "hospital",
    "ward",
    "discharge_disposition",
    "readmission_reason",    
    "primary_diagnosis_desc",
    "primary_diagnosis_icd"
]
df_adm[cat_cols] = df_adm[cat_cols].astype("category")

#### Standardizing the data types of diagnosis dataset

In [5]:
df_diag["diagnosis_rank"] = df_diag["diagnosis_rank"].astype(int)
df_diag["poa_flag"] = df_diag["poa_flag"].astype("category")

df_diag["icd10_code"] = df_diag["icd10_code"].astype("category")

#### Standardizing the data types of laboratory dataset

In [6]:
df_lab["drawn_at"] = pd.to_datetime(df_lab["drawn_at"])

df_lab["value"] = pd.to_numeric(df_lab["value"], errors="coerce")
df_lab["reference_low"] = pd.to_numeric(df_lab["reference_low"], errors="coerce")
df_lab["reference_high"] = pd.to_numeric(df_lab["reference_high"], errors="coerce")

df_lab["flag"] = df_lab["flag"].astype("category")
df_lab["test_name"] = df_lab["test_name"].astype("category")

#### Standardizing the data types of the emergency department ED dataset

In [7]:
df_ed["arrival_datetime"] = pd.to_datetime(df_ed["arrival_datetime"])
df_ed["departure_datetime"] = pd.to_datetime(df_ed["departure_datetime"])

numeric_cols = [
    "triage_level",
    "wait_time_minutes",
    "door_to_doctor_min",
    "ed_los_minutes",
    "hour_of_arrival",
    "month"
]

df_ed[numeric_cols] = df_ed[numeric_cols].apply(pd.to_numeric)

binary_cols = [
    "imaging_ordered",
    "labs_ordered",
    "iv_access",
    "admitted_from_ed",
    "left_ama"
]

df_ed[binary_cols] = df_ed[binary_cols].astype(int)

df_ed["triage_category"] = df_ed["triage_category"].astype("category")
df_ed["disposition"] = df_ed["disposition"].astype("category")

#### Standardizing the data types of the medicationm ddataset

In [8]:
df_meds["start_datetime"] = pd.to_datetime(df_meds["start_datetime"])
df_meds["end_datetime"] = pd.to_datetime(df_meds["end_datetime"])

df_meds["is_high_alert"] = df_meds["is_high_alert"].astype(int)

df_meds["drug_class"] = df_meds["drug_class"].astype("category")
df_meds["route"] = df_meds["route"].astype("category")
df_meds["frequency"] = df_meds["frequency"].astype("category")

#### Standardizing the vitals data types 

In [9]:
df_vitals["measured_at"] = pd.to_datetime(df_vitals["measured_at"])

num_vitals = [
    "systolic_bp",
    "diastolic_bp",
    "heart_rate",
    "respiratory_rate",
    "temperature_c",
    "spo2_percent",
    "gcs_score",
    "news2_score",
    "pain_score",
    "weight_kg",
    "height_cm"
]
df_vitals[num_vitals] = df_vitals[num_vitals].apply(pd.to_numeric)

#### Final Standardization of all the data sets

In [10]:
def standardise(df):
    df.columns = df.columns.str.lower().str.strip()
    return df

dfs = [df_adm, df_diag, df_lab, df_ed, df_pat, df_meds, df_vitals, df_readmit]
      

df_adm, df_diag, df_lab, df_ed, df_pat, df_meds, df_vitals, df_readmit = \
    [standardise(d) for d in dfs]

#### Preventing data leakage

In [11]:
leak_cols = [
    "readmission_date",
    "days_to_readmission",
    "readmission_reason",
    "same_diagnosis",
    "avoided_if_discharged_better"
]

df_readmit = df_readmit.drop(
    columns=[c for c in leak_cols if c in df_readmit.columns],
    errors="ignore"
)

#### Standardise Keys and Dates

In [12]:
DATE_COLS = {
    "df_adm": ["admission_date", "discharge_date"],
    "df_ed": ["arrival_datetime", "departure_datetime"],
    "df_lab": ["drawn_at"],
    "df_meds": ["start_datetime", "end_datetime"],
    "df_vitals": ["measured_at"],
}

In [13]:
TABLES = {
    "df_adm": df_adm,
    "df_ed": df_ed,
    "df_lab": df_lab,
    "df_meds": df_meds,
    "df_vitals": df_vitals,
}

for name, cols in DATE_COLS.items():
    df = TABLES[name]

    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

#### Standardizing the  readmision dataset key

In [14]:
df_readmit["admission_id"] = df_readmit["original_admission_id"].astype(str)
df_adm["admission_id"] = df_adm["admission_id"].astype(str)

In [15]:
readmit_30d = (
    df_readmit
    .assign(readmitted_within_30days=1)
    [["admission_id","readmitted_within_30days"]]
)

df_ml = df_adm.merge(readmit_30d,
                     on="admission_id",
                     how="left")

df_ml["readmitted_within_30days"] = \
    df_ml["readmitted_within_30days"].fillna(0)

#### Build target Variable (Target must come ONLY from readmission table)

##### Merge Patients Master Dataset

In [16]:
df_ml = df_ml.merge(df_pat, on="patient_id", how="left")
df_ml.shape

(8500, 49)

#### Target variable

In [17]:
df_adm["readmitted_within_30-days"] = \
    df_adm["readmitted_within_30d"].astype(int)

#### Aggregate transactional datasets at patient's level

##### Diagnoses Features

In [18]:
diag_features = (
    df_diag.groupby("patient_id")
    .agg(
        num_diagnoses=("icd10_code","count"),
        num_comorbidities=("icd10_code","nunique")
    )
    .reset_index()
)


##### Laboratory features

In [19]:
df_lab["abnormal"] = (
    (df_lab["value"] < df_lab["reference_low"]) |
    (df_lab["value"] > df_lab["reference_high"])
)

lab_features = (
    df_lab.groupby("patient_id")
    .agg(
        num_labs=("test_name","count"),
        abnormal_labs=("abnormal","sum"),
        avg_lab_value=("value","mean")
    )
    .reset_index()
)

##### Medication Features

In [20]:
med_features = (
    df_meds.groupby("patient_id")
    .agg(
        num_medications=("drug_name","count"),
        num_drug_classes=("drug_class","nunique"),
        high_alert_meds=("is_high_alert","sum")
    )
    .reset_index()
)

##### Vital Signs Severity

In [21]:
vital_features = (
    df_vitals.groupby("patient_id")
    .agg(
        max_news2=("news2_score","max"),
        avg_hr=("heart_rate","mean"),
        avg_spo2=("spo2_percent","mean"),
        temp_max=("temperature_c","max")
    )
    .reset_index()
)

##### Emergency Department Compensation

In [22]:
ed_features = (
    df_ed.groupby("patient_id")
    .agg(
        ed_visits_30d=("ed_visit_id","count"),
        avg_wait_time=("wait_time_minutes","mean")
    )
    .reset_index()
)

#### Previous Admission History

In [23]:
prev_adm = (
    df_adm.groupby("patient_id")
    .agg(prev_admissions=("admission_id","count"))
    .reset_index()
)

#### Merging everythin with the anchor dataset

In [24]:
#df_ml = df_adm.copy()

for table in [
    diag_features,
    lab_features,
    med_features,
    vital_features,
    ed_features,
    prev_adm
]:
    df_ml = df_ml.merge(table, on="patient_id", how="left")

In [25]:
df_ml.shape

(8500, 64)

#### Clinical Complexity index

In [26]:
df_ml["clinical_complexity"] = (
    df_ml["num_comorbidities"]
    + df_ml["high_alert_meds"]
    + df_ml["icu_admitted"]
    + df_ml["prev_admissions"]
)

#### Utilization Index

In [ ]:
df_ml["utilisation_score"] = (
    df_ml["ed_visits_30d"]
    + df_ml["num_labs"]
    + df_ml["num_medications"]
)

: 

#### Physiologcal risk

In [28]:
df_ml["physiology_risk"] = (
    df_ml["max_news2"]
    + df_ml["temp_max"]
)

#### Resource Intensity Score

In [29]:
df_ml["resource_intensity"] = (
    df_ml["icu_days"]
    + df_ml["num_procedures"]
    + df_ml["num_labs"]
    + df_ml["num_medications"]
)

#### Care Complexity Score

In [30]:
df_ml["care_complexity"] = (
    df_ml["icu_admitted"]
    + df_ml["high_alert_meds"]
    + df_ml["num_comorbidities"]
)

#### Length of Stay Risk

In [31]:
df_ml["los_risk"] = np.log1p(df_ml["length_of_stay_days"])

#### Final Dataset Check

In [32]:
print(df_ml.shape)
df_ml.info()

(8500, 70)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 70 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   admission_id                8500 non-null   object        
 1   patient_id                  8500 non-null   object        
 2   admission_date              8500 non-null   datetime64[ns]
 3   discharge_date              8500 non-null   datetime64[ns]
 4   length_of_stay_days         8500 non-null   float64       
 5   admission_type              8500 non-null   category      
 6   admission_source            8500 non-null   category      
 7   hospital                    8500 non-null   category      
 8   ward                        8500 non-null   category      
 9   primary_diagnosis_icd       8500 non-null   category      
 10  primary_diagnosis_desc      8500 non-null   category      
 11  drg_code                    8500 non-null   i

#### Saving the Dataset

In [33]:
df_ml.to_csv("read_ml.csv", index=False)

print("✅ read_ml.csv successfully created")

✅ read_ml.csv successfully created


#### Sort Chronologically so as to add historical feature to our dataset

In [34]:
df_adm = df_adm.sort_values(["patient_id","admission_date"])
df_ed["arrival_datetime"] = pd.to_datetime(df_ed["arrival_datetime"])
df_lab["drawn_at"] = pd.to_datetime(df_lab["drawn_at"])
df_meds["start_datetime"] = pd.to_datetime(df_meds["start_datetime"])

#### Helper function - Rolling History Generator

#### 90 days History feature

In [35]:
def rolling_admissions(df, window_days):

    results = []

    for pid, group in df.groupby("patient_id"):
        group = group.sort_values("admission_date")

        dates = group["admission_date"]

        counts = [
            ((dates >= d - pd.Timedelta(days=window_days)) &
             (dates < d)).sum()
            for d in dates
        ]

        group[f"adm_last_{window_days}d"] = counts
        results.append(group)

    return pd.concat(results)

#####  90-Days Rolling Admission History

In [36]:
df_ml = rolling_admissions(df_ml, 90)

#### 360-Day Rolling Admission History

In [37]:
df_ml = rolling_admissions(df_ml, 360)

##### Emergency department ED, visit last 90 & 360 days 

In [38]:
def rolling_ed_visits(adm, ed, window):

    counts = []

    for _, row in adm.iterrows():

        pid = row["patient_id"]
        adm_date = row["admission_date"]

        n = ed[
            (ed.patient_id == pid) &
            (ed.arrival_datetime >= adm_date - pd.Timedelta(days=window)) &
            (ed.arrival_datetime < adm_date)
        ].shape[0]

        counts.append(n)

    return counts

In [39]:
df_ml["ed_visits_last_90d"] = rolling_ed_visits(df_ml, df_ed, 90)
df_ml["ed_visits_last_360d"] = rolling_ed_visits(df_ml, df_ed, 360)

In [40]:
df_ml.shape

(8500, 74)

#### Extending 90-day and 360-day rolling windows to labs and medications

In [41]:
df_lab["drawn_at"] = pd.to_datetime(df_lab["drawn_at"])
df_meds["start_datetime"] = pd.to_datetime(df_meds["start_datetime"])

#### Generic Rolling feature function

In [42]:
def rolling_count_feature(
    admissions,
    events,
    event_time_col,
    feature_name,
    window_days
):

    counts = []

    for _, row in admissions.iterrows():

        pid = row["patient_id"]
        adm_date = row["admission_date"]

        n = events[
            (events.patient_id == pid) &
            (events[event_time_col] >= adm_date - pd.Timedelta(days=window_days)) &
            (events[event_time_col] < adm_date)
        ].shape[0]

        counts.append(n)

    return counts

##### Labs, last 90 days

In [43]:
df_ml["labs_last_90d"] = rolling_count_feature(
    df_ml,
    df_lab,
    "drawn_at",
    "labs",
    90
)


#### labs last 360-Days

In [44]:
df_ml["labs_360d"] = rolling_count_feature(
    df_ml,
    df_lab,
    "drawn_at",
    "labs",
    360
)

In [45]:
df_lab["abnormal_flag"] = (
    (df_lab["value"] < df_lab["reference_low"]) |
    (df_lab["value"] > df_lab["reference_high"])
).astype(int)

#### Rolling Abnormal labs

In [46]:
def rolling_abnormal_labs(adm, labs, window):

    counts = []

    for _, row in adm.iterrows():

        pid = row["patient_id"]
        adm_date = row["admission_date"]

        n = labs[
            (labs.patient_id == pid) &
            (labs.drawn_at >= adm_date - pd.Timedelta(days=window)) &
            (labs.drawn_at < adm_date) &
            (labs.abnormal_flag == 1)
        ].shape[0]

        counts.append(n)

    return counts

In [47]:
df_ml["abnormal_labs_90d"] = rolling_abnormal_labs(df_ml, df_lab, 90)
df_ml["abnormal_labs_360d"] = rolling_abnormal_labs(df_ml, df_lab, 360)

##### Medication exposure counts in the last 90-days & 360-days

In [48]:
df_ml["meds_90d"] = rolling_count_feature(
    df_ml,
    df_meds,
    "start_datetime",
    "meds",
    90
)

df_ml["meds_360d"] = rolling_count_feature(
    df_ml,
    df_meds,
    "start_datetime",
    "meds",
    360
)

#### High Alert medication hisory

In [49]:
def rolling_high_alert(adm, meds, window):

    counts = []

    for _, row in adm.iterrows():

        pid = row["patient_id"]
        adm_date = row["admission_date"]

        n = meds[
            (meds.patient_id == pid) &
            (meds.start_datetime >= adm_date - pd.Timedelta(days=window)) &
            (meds.start_datetime < adm_date) &
            (meds.is_high_alert == 1)
        ].shape[0]

        counts.append(n)

    return counts

In [50]:
df_ml["high_alert_90d"] = rolling_high_alert(df_ml, df_meds, 90)
df_ml["high_alert_360d"] = rolling_high_alert(df_ml, df_meds, 360)

#### Create a Hospital-Level Composite Risk Signal

In [51]:
df_ml["clinical_instability_score"] = (
      df_ml["adm_last_90d"]
    + df_ml["ed_visits_last_90d"]
    + df_ml["abnormal_labs_90d"]
    + df_ml["high_alert_90d"]
)

#### Time Since Last Discharge

In [52]:
df_ml["time_since_last_discharge"] = (
    df_ml.groupby("patient_id")["discharge_date"]
    .shift(1)
)

df_ml["time_since_last_discharge"] = (
    df_ml["admission_date"]
    - df_ml["time_since_last_discharge"]
).dt.days.fillna(999)

#### Admission Velocity (Healthcare dependency)

In [53]:
df_ml["admission_velocity"] = (
    df_ml["adm_last_90d"] /
    (df_ml["adm_last_360d"] + 1)
)

#### Pharmacy Index

In [54]:
df_ml["polypharmacy_index"] = (
    df_ml["num_medications"] +
    df_ml["high_alert_meds"]
)

#### Instability Score

In [55]:
df_ml["instability_score"] = (
    df_ml["max_news2"]
    + df_ml["abnormal_labs"]
    + df_ml["ed_visits_last_90d"]
)

#### Chronic Utilization Score

In [56]:
df_ml["chronic_utilisation_score"] = (
    df_ml["adm_last_360d"]
    + df_ml["ed_visits_last_360d"]
    + df_ml["prev_admissions"]
)

#### Care Fragmentation

In [57]:
df_ml["care_fragmentation"] = (
    df_ml.groupby("patient_id")["hospital"]
    .transform("nunique")
)

In [58]:
df_ml.shape

(8500, 89)

In [71]:
drop_cols=["admission_id",
    "readmission_date",
    "admission_date",
    "attending_physician_id",
    "readmitted_30d",
    "total_charges_usd",
    "insurance_paid_usd",
    "readmission_reason",
    "mrn",
    "ed_visits_last_90d_y",
    "readmitted-within_30d",
    "adm_last_90d_y",
    "ed_visits_last_360d_y",
    "labs_last_90d_y",
    "meds_90d_y",
    "meds_360d_y",
    "adm_last_360d_y",
    "admission_date",
    "adm_velocity_y",
    "first_name",
    "last_name",
    "date_of_birth",
    "age",
    "zip_code",
    "admission_id",
    "time_since_last_discharge_y",
    "admission_velocity_y",
    "language_preference",
    "registeration_date",
    "readmitted_within_30d"
    
]
df_ml = df_ml.drop(columns=[c for c in drop_cols if c in df_ml.columns])


#### Save Final dataset

In [73]:
df_ml.to_csv("datasets/processed/read_ml.csv", index=False)

print("✅ read_ml.csv with rolling history features created")

✅ read_ml.csv with rolling history features created


In [72]:
df_ml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8500 entries, 0 to 8499
Data columns (total 75 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   patient_id                   8500 non-null   object        
 1   discharge_date               8500 non-null   datetime64[ns]
 2   length_of_stay_days          8500 non-null   float64       
 3   admission_type               8500 non-null   category      
 4   admission_source             8500 non-null   category      
 5   hospital                     8500 non-null   category      
 6   ward                         8500 non-null   category      
 7   primary_diagnosis_icd        8500 non-null   category      
 8   primary_diagnosis_desc       8500 non-null   category      
 9   drg_code                     8500 non-null   int64         
 10  icu_admitted                 8500 non-null   int64         
 11  icu_days                     8500 non-null 